# Запуск и проверка Predictive Maintenance MLOps System (отчет)

Этот ноутбук фиксирует воспроизводимый запуск MLOps-системы для задачи предиктивного обслуживания промышленного оборудования.

Цель отчёта — показать, что проект можно поднять из репозитория, проверить тестами, запустить инфраструктуру, открыть основные UI, выполнить inference-запросы, проверить мониторинг, SLO rules, data drift report и canary traffic switching.

Проект включает:

- FastAPI inference service;
- Feast Feature Store;
- Redis Online Store;
- PostgreSQL;
- MLflow Tracking Server и Model Registry;
- Airflow orchestration;
- Prometheus monitoring;
- Grafana dashboard;
- Node Exporter infrastructure monitoring;
- Evidently data drift report;
- Nginx-based canary gateway;
- Docker Compose и Ansible как Infrastructure as Code.

Все команды ниже предполагают запуск из корня репозитория `predictive-maintenance-mlops`.

# Порты 

```text
8000   FastAPI API
8010   Canary Gateway
5050   MLflow UI
8081   Airflow UI
9090   Prometheus
9100   Node Exporter
3000   Grafana
15432  PostgreSQL
16379  Redis
```

# Корень проекта

In [ ]:
%env PROJECT_ROOT=/Users/perceivery/Desktop/predictive-maintenance-mlops

# Проверка файлов проекта

In [ ]:
%%bash
cd "$PROJECT_ROOT"
pwd
ls -la | head

# Проверка состояния репозитория

Перед запуском сервисов фиксируем состояние Git-репозитория.

In [ ]:
%%bash
cd "$PROJECT_ROOT"
git status
git log --oneline -8

# Проверка локального окружения

Проверяем, что доступны Python, Docker, Docker Compose и Makefile. Эти инструменты нужны для запуска тестов, сборки контейнеров и поднятия MLOps-инфраструктуры.

In [ ]:
%%bash
cd "$PROJECT_ROOT"
python --version
docker --version
docker compose version
make --version | head -n 1

# Создание venv  и установка зависимостей

Создаём локальное Python-окружение `.venv` и устанавливаем зависимости проекта из `requirements.txt`.

В проекте установка обёрнута в Makefile-команду

```bash
make install
```

In [ ]:
%%bash
cd "$PROJECT_ROOT"
if [ ! -d ".venv" ]; then
  python3 -m venv .venv
fi

. .venv/bin/activate

In [ ]:
%%bash
cd "$PROJECT_ROOT"
make install

# Запуск тестов проекта

После проверки окружения запускаем автоматические тесты проекта.

Команда `make test` использует pytest и проверяет базовую работоспособность API-логики и вспомогательных компонентов.

Ожидаемый результат:

```text
9 passed
```

In [ ]:
%%bash
cd "$PROJECT_ROOT"
make test

# Проверка Docker Compose конфигурации основного контура

Перед запуском сервисов проверяем, что основной Docker Compose файл синтаксически корректен и может быть собран Docker Compose.

Основной контур описан в файле:

```text
infra/docker-compose.yml
```
В него входят PostgreSQL, Redis, MLflow, FastAPI, Airflow, Prometheus, Grafana и Node Exporter.

In [ ]:
%%bash
cd "$PROJECT_ROOT"
docker compose -f infra/docker-compose.yml config >/tmp/main-compose-config.txt
echo "Main Docker Compose config is valid."

# Проверка Docker Compose конфигурации canary-контура

Canary-контур описан в отдельном файле:

```text
infra/docker-compose.canary.yml
```

Он поднимает два экземпляра inference service:

- stable;
- canary;

и Nginx gateway, который переключает трафик между ними.

In [ ]:
%%bash
cd "$PROJECT_ROOT"
docker compose -f infra/docker-compose.canary.yml config >/tmp/canary-compose-config.txt
echo "Canary Docker Compose config is valid."

# Запуск основного MLOps-контура

Поднимаем основной Docker Compose контур из файла:

```text
infra/docker-compose.yml
```

В этом контуре запускаются основные сервисы проекта: PostgreSQL, Redis, MLflow, FastAPI, Airflow, Prometheus, Grafana и Node Exporter.

Команда выполняет сборку образов при необходимости и запускает контейнеры в фоне.

In [ ]:
%%bash
cd "$PROJECT_ROOT"
docker compose -f infra/docker-compose.yml up -d --build

# Проверка статусов контейнеров основного контура

После запуска проверяем состояние контейнеров. Для backend-сервисов важно увидеть, что контейнеры находятся в статусе `Up`, а сервисы с healthcheck имеют статус `healthy`.

Эта проверка нужна для подтверждения, что заявленные компоненты MLOps-системы действительно запущены.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.yml ps

# Проверка FastAPI health endpoint

Проверяем, что inference API доступен по HTTP и возвращает статус сервиса.

Endpoint: GET /health


Ожидаемый результат: HTTP-запрос должен вернуть JSON со статусом ok, признаком загруженной модели и deployment track.

In [ ]:
%%bash

cd "$PROJECT_ROOT"
curl -s http://127.0.0.1:8000/health

# Проверка информации о production-модели

Проверяем endpoint: GET /model/info

Он показывает, какую модель FastAPI service загрузил из MLflow Model Registry.

Ожидаемый результат:

- model_name = predictive-maintenance-model;
- model_alias = champion;
- status = loaded.

In [ ]:
%%bash

cd "$PROJECT_ROOT"
curl -s http://127.0.0.1:8000/model/info

# Проверка inference endpoint `/predict`

Проверяем endpoint: POST /predict

Этот endpoint принимает полный набор признаков в request body и возвращает прогноз отказа оборудования, вероятность отказа, уровень риска и рекомендуемое действие.

Ожидаемый результат: HTTP-запрос должен вернуть JSON с полями failure_probability, prediction, risk_level, recommended_action, model_name и model_alias.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "air_temperature_k": 298.1,
    "process_temperature_k": 308.6,
    "rotational_speed_rpm": 1551,
    "torque_nm": 42.8,
    "tool_wear_min": 120,
    "machine_type_H": 0,
    "machine_type_L": 0,
    "machine_type_M": 1
  }'

# Проверка Feast Feature Store

Feast используется для управления признаками модели.

В проекте проверяются два сценария работы Feature Store:

- offline retrieval  — получение исторических признаков для обучения
- online retrieval   — получение online-признаков из Redis для inference
  
Проверка выполняется скриптом: pipelines/check_feature_store.py

Скрипт проверяет offline retrieval, материализацию признаков в Redis Online Store и online retrieval по machine_id.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

. .venv/bin/activate
FEAST_REDIS_CONNECTION_STRING=localhost:16379 python pipelines/check_feature_store.py

# Проверка production-like inference через Feast Redis

Проверяем endpoint: POST /predict/from-feature-store

Этот endpoint принимает только machine_id, получает online-признаки из Feast Redis Online Store и затем выполняет inference champion-моделью из MLflow Model Registry.

Эта проверка важна, потому что она показывает не просто ручную передачу признаков в API, а прод путь:

**machine_id -> Feast Redis Online Store -> FastAPI -> MLflow champion model -> prediction**

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s -X POST http://127.0.0.1:8000/predict/from-feature-store \
  -H "Content-Type: application/json" \
  -d '{"machine_id": 1}'

# Проверка FastAPI Prometheus metrics endpoint

FastAPI service отдаёт технические метрики на endpoint: GET /metrics

Prometheus использует этот endpoint для сбора метрик inference service.

Проверяем наличие основных метрик:

- predict_requests_total;
- predict_errors_total;
- predict_latency_seconds;
- feature_retrieval_requests_total;
- feature_retrieval_errors_total;
- feature_retrieval_latency_seconds.

In [ ]:
%%bash

cd "$PROJECT_ROOT"
curl -s http://127.0.0.1:8000/metrics | grep -E "predict_requests_total|predict_errors_total|predict_latency_seconds|feature_retrieval_requests_total|feature_retrieval_errors_total|feature_retrieval_latency_seconds" | head -n 40

# Проверка Prometheus health endpoint

Prometheus используется для сбора технических метрик FastAPI service, Node Exporter и самого Prometheus.

Проверяем health endpoint: GET /-/healthy

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:9090/-/healthy

# Проверка Prometheus targets

Проверяем, что Prometheus видит основные адреса сервисов:

- `predictive-maintenance-api` - метрики FastAPI: latency, ошибки, количество запросов, feature retrieval.
- `node-exporter` - метрики машины/инфраструктуры: CPU, память, диск итд
- `prometheus` — метрики самого Prometheus.

Нужно показать, что мониторинг не только запущен, но и реально собирает метрики с сервисов.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s "http://127.0.0.1:9090/api/v1/targets" \
  | python -m json.tool \
  | grep -E '"job"|"health"|"scrapeUrl"' \
  | head -n 40

# Проверка Prometheus alert rules / SLO rules

Проверяем, что Prometheus загрузил alert rules, связанные с SLO inference service.

В проекте используются правила для контроля:

- высокой latency;
- высокого error rate;
- недоступности API;
- ошибок online feature retrieval из Feast Redis.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s "http://127.0.0.1:9090/api/v1/rules" \
  | python -m json.tool \
  | grep -E '"name": "PredictiveMaintenance|state|health|query' \
  | head -n 80

# Проверка Grafana health endpoint

Grafana используется для визуализации метрик Prometheus.

Проверяем health endpoint Grafana: GET /api/health

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:3000/api/health

# Проверка MLflow Tracking Server

MLflow используется для отслеживания экспериментов, хранения метрик обучения, артефактов моделей и ведения реестра моделей.

Проверяем, что MLflow-сервер доступен по HTTP.

В локальном Docker Compose контуре веб-интерфейс MLflow открыт на порту: http://127.0.0.1:5050

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s -I http://127.0.0.1:5050 | head

# Проверка Airflow

Airflow используется для оркестрации пайплайна обучения и продвижения модели.

Проверяем служебный endpoint Airflow: GET /health

В локальном Docker Compose контуре веб-интерфейс Airflow открыт на порту: http://127.0.0.1:8081

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:8081/health

# Проверка DAG в Airflow

Проверяем, что Airflow видит DAG пайплайна обучения: predictive_maintenance_training_pipeline

Этот DAG отвечает за запуск этапов подготовки данных, построения признаков, проверки Feature Store, обучения моделей и продвижения лучшей модели.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.yml exec airflow-scheduler \
  airflow dags list | grep predictive_maintenance_training_pipeline

# Запуск Airflow DAG

Запускаем DAG `predictive_maintenance_training_pipeline`.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.yml exec airflow-scheduler \
  airflow dags trigger predictive_maintenance_training_pipeline

# Проверка статуса запущенного DAG

После запуска DAG проверяем его состояние.  
Сначала DAG может быть в статусе `queued` или `running`. После завершения успешного пайплайна ожидаемый статус `success`.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

sleep 20

docker compose -f infra/docker-compose.yml exec airflow-scheduler \
  airflow dags list-runs -d predictive_maintenance_training_pipeline | head -n 20

# Повторная проверка production-модели после запуска DAG

После успешного запуска Airflow DAG повторно проверяем endpoint `/model/info`.

Это подтвердит, что FastAPI service видит production-модель из MLflow Model Registry и загружает её по alias `champion`.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:8000/model/info

# Проверка Node Exporter

Node Exporter используется для сбора инфраструктурных метрик машины таких как CPU, памяти, диска и других системных показателей.

Проверяем, что Node Exporter отдаёт метрики на endpoint: GET /metrics

В локальном контуре Node Exporter доступен на порту 9100.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:9100/metrics | grep -E "node_cpu_seconds_total|node_memory|node_filesystem" | head -n 20

# Проверка инфраструктурных метрик через Prometheus

Теперь проверяем не только сам Node Exporter, но и то что Prometheus успешно собирает его метрики.

Для этого запрашиваем метрику `node_cpu_seconds_total` через API Prometheus.

In [ ]:
%%bash
cd "$PROJECT_ROOT"
curl -s "http://127.0.0.1:9090/api/v1/query?query=node_cpu_seconds_total" \
  | python -m json.tool \
  | grep -E '"status"|"__name__"|"job"|"instance"' \
  | head -n 20

# Проверка data drift report через Evidently

Evidently используется для контроля дрифта входных данных.

Скрипт `pipelines/check_data_drift.py` сравнивает reference dataset и current dataset, строит HTML-отчёт и сохраняет JSON summary.

Ожидаемые артефакты:

```text
reports/evidently/data_drift_report.html
reports/evidently/data_drift_summary.json
```

In [ ]:
%%bash
cd "$PROJECT_ROOT"

. .venv/bin/activate
make drift-check

# Проверка артефактов Evidently

Проверяем, что после запуска drift-check были созданы оба артефакта

- HTML-отчёт для визуального анализа;
- JSON summary для машинно-читаемого результата.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

ls -lh reports/evidently/data_drift_report.html
ls -lh reports/evidently/data_drift_summary.json

# Запуск canary-контура

Теперь запускаем отдельный canary-контур из файла: infra/docker-compose.canary.yml

Он поднимает:

- stable inference service;
- canary inference service;
- Nginx gateway на порту 8010.

Canary gateway нужен для демонстрации постепенного переключения трафика между stable и canary API-сервисами.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.canary.yml up -d --build

# Проверка статусов canary-контура

Проверяем, что поднялись оба экземпляра inference service:

- `stable`;
- `canary`;

а также Nginx gateway, который принимает внешний трафик на порту `8010`.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.canary.yml ps

# Проверка canary gateway

Проверяем, что Nginx доступен на порту `8010` и проксирует запросы к inference service.

Endpoint: GET /health

Ожидаемый результат: JSON с status = ok и полем deployment_track, которое показывает какой backend ответил на запрос (stable или canary).

In [ ]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:8010/health

# Проверка распределения трафика через canary

Проверяем, как Nginx-шлюз распределяет запросы между двумя версиями сервиса:

- `stable` — стабильная версия сервиса;
- `canary` — тестовая версия сервиса.

Скрипт `scripts/check_canary_distribution.sh` выполняет несколько запросов к endpoint `/health` через Nginx-шлюз и считает, какая версия сервиса ответила на запрос.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

N=50 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

# Переключение трафика в режим 50/50

Проверяем механизм canary переключения.

Скрипт `scripts/switch_canary_50_50.sh` меняет конфигурацию Nginx-шлюза так, чтобы примерно половина запросов шла в `stable`, а половина в `canary`.

После переключения повторно проверим распределение запросов.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

scripts/switch_canary_50_50.sh
sleep 5
N=50 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

# Переключение 100% трафика на canary

Проверяем сценарий полного переключения трафика на canary версию сервиса.

Скрипт `scripts/switch_canary_100.sh` меняет конфигурацию Nginx-шлюза так, чтобы все запросы через порт `8010` направлялись в `canary`.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

scripts/switch_canary_100.sh
sleep 5
N=20 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

# Rollback возврат трафика на stable

Проверяем сценарий отката.

Скрипт `scripts/rollback_canary_to_stable.sh` возвращает конфигурацию Nginx-шлюза в безопасный режим то есть все запросы снова направляются в `stable`.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

scripts/rollback_canary_to_stable.sh
sleep 5
N=20 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

# Итоговая проверка canary-контура после rollback

После проверки переключения трафика повторно проверяем состояние canary-контейнеров.

Важно чтобы после rollback оба сервиса `stable` и `canary` оставались работоспособными, а Nginx-шлюз продолжал отвечать на запросы.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.canary.yml ps
curl -s http://127.0.0.1:8010/health

# Проверка CI/CD через GitHub Actions

GitHub Actions используется как CI-контур проекта.

В CI автоматически проверяется, что проект устанавливается и проходит тесты. Это дополняет локальную проверку `make test` то есть код проверяется не только на машине разработчика, но и в удалённом CI-окружении GitHub.

In [ ]:
%%bash
cd "$PROJECT_ROOT"

ls -la .github/workflows
cat .github/workflows/ci.yml

CI workflow находится в файле `.github/workflows/ci.yml`.

Он запускается при `push` и `pull_request` в ветку `main`. В workflow выполняются установка Python 3.12, установка зависимостей из `requirements.txt` и запуск тестов через `pytest`.

<h3>СI</h3>
<img src="../screenshots/10_github_actions_success.png" alt="MLflow experiments" width="900">

# Проверка Ansible Infrastructure as Code

Ansible используется как Infrastructure as Code слой для развёртывания проекта на виртуальной машине.

Docker Compose описывает состав контейнеров, а Ansible описывает воспроизводимый процесс подготовки сервера и запуска проекта:

```text
пустая VM
-> установка системных пакетов
-> установка и запуск Docker
-> копирование проекта на VM
-> сборка Docker-образов
-> запуск MLOps-инфраструктуры
-> обучение и promotion модели
-> проверка health endpoints

Основные файлы Ansible-контура:
- ansible/inventory.ini
- ansible/group_vars/all.yml
- ansible/playbook.yml
- ansible/README.md
```

In [ ]:
%%bash
cd "$PROJECT_ROOT"

ls -la ansible/
echo "----- group_vars -----"
ls -la ansible/group_vars/

python - <<'PY'
import yaml

for file in [
    "ansible/group_vars/all.yml",
    "ansible/playbook.yml",
]:
    with open(file, "r", encoding="utf-8") as f:
        yaml.safe_load(f)
    print(f"YAML OK: {file}")
PY

Фактический запуск на виртуальной машине выполняется командой:

```bash
ansible-playbook -i ansible/inventory.ini ansible/playbook.yml
```

# Дадим нагрузку на инференс

In [ ]:
%%bash
cd "$PROJECT_ROOT"

echo "Starting inference load for 120 seconds..."
echo "Endpoint: /predict/from-feature-store"

end=$((SECONDS + 120))

while [ $SECONDS -lt $end ]; do
  for i in $(seq 1 20); do
    curl -s -X POST http://127.0.0.1:8000/predict/from-feature-store \
      -H "Content-Type: application/json" \
      -d '{"machine_id": 1}' >/dev/null &
  done

  wait
  sleep 1
done

echo "Inference load finished."

curl -s http://127.0.0.1:8000/metrics \
  | grep -E "^predict_requests_total|^predict_errors_total|^feature_retrieval_requests_total|^feature_retrieval_errors_total"

<h3>Grafana dashboard</h3>
<img src="../screenshots/11_grafana_dashboard_after_load.png" alt="Grafana dashboard" width="900">

# Веб-интерфейсы для ручной проверки

После запуска сервисов доступны следующие веб-интерфейсы:

| Компонент | Адрес | Что проверить |
|---|---|---|
| FastAPI | `http://127.0.0.1:8000/docs` | Наличие endpoint-ов `/health`, `/model/info`, `/predict`, `/predict/from-feature-store`, `/metrics` |
| MLflow | `http://127.0.0.1:5050` | Эксперименты, запуски обучения, зарегистрированная модель `predictive-maintenance-model` |
| Airflow | `http://127.0.0.1:8081` | DAG `predictive_maintenance_training_pipeline` и последний запуск со статусом `success` |
| Prometheus targets | `http://127.0.0.1:9090/targets` | Targets `predictive-maintenance-api`, `node-exporter`, `prometheus` в состоянии `UP` |
| Prometheus rules | `http://127.0.0.1:9090/rules` | Правила `PredictiveMaintenanceHighLatency`, `PredictiveMaintenanceHighErrorRate`, `PredictiveMaintenanceApiDown`, `PredictiveMaintenanceFeatureRetrievalErrors` |
| Grafana | `http://127.0.0.1:3000` | Dashboard `Predictive Maintenance API` |
| Evidently report | `reports/evidently/data_drift_report.html` | HTML-отчёт по drift данных |
| Canary gateway | `http://127.0.0.1:8010/health` | Ответ от `stable` после rollback |

Логин и пароль для Airflow, Grafana:

```text
admin / admin
```

<h2>Скриншоты работающей системы</h2>

<h3>MLflow: эксперименты</h3>
<img src="../screenshots/01_mlflow_experiments.png" alt="MLflow experiments" width="900">

<h3>MLflow Model Registry: champion-модель</h3>
<img src="../screenshots/02_mlflow_model_registry_champion.png" alt="MLflow champion model" width="900">

<h3>Airflow: успешный запуск DAG</h3>
<img src="../screenshots/03_airflow_dag_success.png" alt="Airflow DAG success" width="900">

<h3>Prometheus targets</h3>
<img src="../screenshots/04_prometheus_targets.png" alt="Prometheus targets" width="900">

<h3>Prometheus rules</h3>
<img src="../screenshots/05_prometheus_rules1.png" alt="Prometheus rules" width="900">
<img src="../screenshots/05_prometheus_rules2.png" alt="Prometheus rules" width="900">

<h3>Grafana dashboard</h3>
<img src="../screenshots/06_grafana_dashboard1.png" alt="Grafana dashboard" width="900">
<img src="../screenshots/06_grafana_dashboard2.png" alt="Grafana dashboard" width="900">

<h3>FastAPI Swagger UI</h3>
<img src="../screenshots/07_fastapi_docs.png" alt="FastAPI docs" width="900">

<h3>Evidently drift report</h3>
<img src="../screenshots/08_evidently_report1.png" alt="Evidently drift report" width="900">
<img src="../screenshots/08_evidently_report2.png" alt="Evidently drift report" width="900">

<h3>Canary traffic switching</h3>
<img src="../screenshots/09_canary.png" alt="Canary traffic distribution" width="900">
